In [2]:
import pandas as pd
import sqlalchemy
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv('/Users/naren/Projects/attrition-payroll-risk/.env')

# Connect to MySQL
engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}"
)

# Test connection
with engine.connect() as conn:
    print("MySQL connected successfully!")

MySQL connected successfully!


In [3]:
# Load processed CSV
df = pd.read_csv('/Users/naren/Projects/attrition-payroll-risk/data/processed/attrition_processed.csv')

# Select only raw columns for employee_raw table
raw_cols = [
    'EmployeeNumber', 'Age', 'Attrition', 'BusinessTravel', 'Department',
    'DistanceFromHome', 'Education', 'EducationField', 'EnvironmentSatisfaction',
    'Gender', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus',
    'MonthlyIncome', 'NumCompaniesWorked', 'OverTime', 'PercentSalaryHike',
    'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel',
    'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance',
    'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion',
    'YearsWithCurrManager'
]

df_raw = df[raw_cols]

# Load to MySQL
df_raw.to_sql('employee_raw', con=engine, if_exists='replace', index=False)
print(f"employee_raw table loaded: {len(df_raw)} rows")

employee_raw table loaded: 1470 rows


In [4]:
# Select engineered payroll feature columns
feature_cols = [
    'EmployeeNumber', 'Attrition_Flag', 'Compensation_Ratio',
    'Hike_Band', 'OverTime_Flag', 'Overtime_LowHike_Risk',
    'Tenure_Per_Level', 'Total_Comp_Score', 'Exp_Pay_Ratio'
]

df_features = df[feature_cols]

# Load to MySQL
df_features.to_sql('employee_features', con=engine, if_exists='replace', index=False)
print(f"employee_features table loaded: {len(df_features)} rows")

employee_features table loaded: 1470 rows


In [5]:
import pickle

# Load saved model, scaler and feature list
with open('/Users/naren/Projects/attrition-payroll-risk/models/attrition_model.pkl', 'rb') as f:
    model = pickle.load(f)

with open('/Users/naren/Projects/attrition-payroll-risk/models/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

with open('/Users/naren/Projects/attrition-payroll-risk/models/feature_cols.pkl', 'rb') as f:
    model_features = pickle.load(f)

# Generate predictions
X = df[model_features]
X_scaled = scaler.transform(X)

df['Attrition_Probability'] = model.predict_proba(X_scaled)[:, 1]
df['Attrition_Prediction'] = model.predict(X_scaled)

# Assign Risk Category based on probability
def risk_category(prob):
    if prob >= 0.7:
        return 'High Risk'
    elif prob >= 0.4:
        return 'Medium Risk'
    else:
        return 'Low Risk'

df['Risk_Category'] = df['Attrition_Probability'].apply(risk_category)

# Select prediction columns
df_predictions = df[['EmployeeNumber', 'Attrition_Probability',
                      'Attrition_Prediction', 'Risk_Category']]

# Load to MySQL
df_predictions.to_sql('employee_predictions', con=engine, if_exists='replace', index=False)
print(f"employee_predictions table loaded: {len(df_predictions)} rows")
print("\nRisk Category Distribution:")
print(df['Risk_Category'].value_counts())

employee_predictions table loaded: 1470 rows

Risk Category Distribution:
Risk_Category
Low Risk       1169
High Risk       207
Medium Risk      94
Name: count, dtype: int64


In [8]:
from sqlalchemy import text

with engine.connect() as conn:
    result = conn.execute(text("SHOW TABLES"))
    tables = result.fetchall()
    print("Tables in attrition_db:")
    for table in tables:
        print(" -", table[0])

    # Row counts
    for table_name in ['employee_raw', 'employee_features', 'employee_predictions']:
        count = conn.execute(text(f"SELECT COUNT(*) FROM {table_name}")).scalar()
        print(f"\n{table_name}: {count} rows")

Tables in attrition_db:
 - employee_features
 - employee_predictions
 - employee_raw

employee_raw: 1470 rows

employee_features: 1470 rows

employee_predictions: 1470 rows


In [9]:
with engine.connect() as conn:
    result = conn.execute(text("SHOW TABLES"))
    tables = result.fetchall()
    print("Tables in attrition_db:")
    for table in tables:
        print(" -", table[0])

    # Row counts
    for table_name in ['employee_raw', 'employee_features', 'employee_predictions']:
        count = conn.execute(text(f"SELECT COUNT(*) FROM {table_name}")).scalar()
        print(f"\n{table_name}: {count} rows")

Tables in attrition_db:
 - employee_features
 - employee_predictions
 - employee_raw

employee_raw: 1470 rows

employee_features: 1470 rows

employee_predictions: 1470 rows


In [10]:
# Test Query: Attrition by Department
query = """
    SELECT
        e.Department,
        COUNT(*) AS Total,
        SUM(f.Attrition_Flag) AS Attrited,
        ROUND(SUM(f.Attrition_Flag) * 100.0 / COUNT(*), 2) AS Attrition_Rate_Pct
    FROM employee_raw e
    JOIN employee_features f ON e.EmployeeNumber = f.EmployeeNumber
    GROUP BY e.Department
    ORDER BY Attrition_Rate_Pct DESC
"""

result = pd.read_sql(query, con=engine)
print("Attrition by Department:")
print(result)

Attrition by Department:
               Department  Total  Attrited  Attrition_Rate_Pct
0                   Sales    446      92.0               20.63
1         Human Resources     63      12.0               19.05
2  Research & Development    961     133.0               13.84
